## Demo Notebook

Drop not needed columns

In [ ]:
# from torch import embedding
# from textprep.vectorizer import *
# from textprep.embed import EmbeddingsHandler
# import pandas as pd

# from utils.download import *
# from pulearn.pubase import *


# download_from_gdrive("deceptive-opinion.csv")

# documents = pd.read_csv("../data/deceptive-opinion.csv")
# documents["deceptive"] = documents["deceptive"].apply(lambda x: 0 if x == "truthful" else 1)
# documents["label"] = documents["deceptive"].copy()
# documents = documents.drop(columns=["hotel", "source", "polarity", "deceptive"])

# documents["sequences"], documents["text"], vectorizer = Vectorizer.from_dataframe (
#     documents, 
#     "text", 
#     "label", 
#     prep_funcs={
#         to_lower: {},
#         to_remove_symbols: {}
#     }
# )

# emb_handler = EmbeddingsHandler(vectorizer.text_vocab, pretrained="fasttext-wiki")
# emb_handler.load_embeddings()

In [ ]:
from pulearn.neural_nets.basenet import DocumentClassifier
from pulearn.neural_nets.data_modules import DeceptiveOpinionsDataModule

import pytorch_lightning as pl


datamodule = DeceptiveOpinionsDataModule()
datamodule.prepare_data()
embeddings = datamodule.embeddings

doc_clf = DocumentClassifier(pretrained_embedding=embeddings)

trainer = pl.Trainer(
    max_epochs=5,
    fast_dev_run=True,
    accelerator="gpu",
    devices=1
)

trainer.fit(doc_clf, datamodule=datamodule)

Isolate all deceptive documents and extract a partial test set of 160 instances

In [ ]:
# deceptive_docs = documents.loc[(documents["label"] == 1)]
# deceptive_docs.reset_index(drop=True, inplace=True)
# indices = extract_sample(deceptive_docs["label"], ratio=0.2, value=1)
# deceptive_set = deceptive_docs.loc[indices]
# deceptive_set

Isolate all non-deceptive documents and extract a partial test set of another 160 instances

In [ ]:
# non_deceptive_docs = documents.loc[(documents["label"] == 0)]
# non_deceptive_docs.reset_index(drop=True, inplace=True)
# indices = extract_sample(non_deceptive_docs["label"], ratio=0.2, value=0)
# non_deceptive_set = non_deceptive_docs.loc[indices]
# non_deceptive_set

Merge above sets to create a unified test set

In [ ]:
# test_set = pd.concat([deceptive_set, non_deceptive_set], ignore_index=True)
# test_set

Accordingly extract a training set and turn it into a PU dataset

In [ ]:
# train_set = documents.loc[~(documents["text"].isin(test_set["text"]))]
# train_set = train_set.reset_index(drop=True)

# indices = extract_sample(train_set["label"], ratio=0.37, value=1)
# train_set["pu-label"] = 0
# train_set.loc[indices, "pu-label"] = 1

# train_set["pu-label"].value_counts()

Create TF-IDF vectors

In [ ]:
# from sklearn.feature_extraction.text import TfidfVectorizer


# X_train = train_set["text"]
# y_train = train_set["pu-label"]

# X_test = test_set["text"]
# y_test = test_set["label"]

# tfidf_input = pd.concat([X_train, X_test])

# vectorizer = TfidfVectorizer()

# vectors = vectorizer.fit_transform(tfidf_input)
# feature_names = vectorizer.get_feature_names_out()
# dense = vectors.todense().tolist()

# X = pd.DataFrame(dense, columns=feature_names)

# train_rows = X_train.shape[0]
# test_rows = X_test.shape[0]

# X_train = X.iloc[:train_rows]
# X_test = X.iloc[-test_rows:]
# X_test = X_test.reset_index(drop=True)

Growing classifier

In [ ]:
# from pulearn.iterative.growing import GrowingClassifier
# from sklearn.metrics import f1_score


# roc_svm = GrowingClassifier("Rocchio", "SVC", params1={"metric": "manhattan"}, params2={"kernel": "linear"})
# roc_svm.fit(X_train, y_train)
# predictions = roc_svm.predict(X_test)

# print("\nF1-score = {:.2f}".format(f1_score(y_test, predictions, average="macro") * 100))

Pruning Classifier -- currently contains a bug which results in an endless loop

Reason: internal 2nd estimator **sometimes** makes only negative predictions & results in while condition never exiting

In [ ]:
# from pulearn.iterative.pruning import PruningClassifier


# roc_svm = PruningClassifier("Rocchio", "SVC", params1={"metric": "manhattan"}, params2={"kernel": "linear"})
# roc_svm.fit(X_train, y_train)
# predictions = roc_svm.predict(X_test)

# print("\nF1-score = {:.2f}".format(f1_score(y_test, predictions, average="macro") * 100))